# Faster R-CNN + ResNet50 + FPN
Dataset: Roboflow (formato COCO) — 2 clases (Buena, Mala)

**Kaggle:** Activar Internet en Settings → Enable Internet

In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

install('roboflow')
install('torchmetrics')

print('Dependencias instaladas')

In [ ]:
import os
import json
import shutil
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import torchvision.transforms.functional as TF

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f'Usando: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# --- TABLA MAESTRA ---
NUM_CLASSES   = 3       # fondo(0) + Buena(1) + Mala(2)
EPOCHS        = 50
BATCH_SIZE    = 2
LEARNING_RATE = 0.0001
WEIGHT_DECAY  = 0.0005
STEP_LR       = 5
GAMMA_LR      = 0.1
WORK_DIR      = '/kaggle/working'
# ---------------------

In [ ]:
from roboflow import Roboflow

# Limpiar descarga anterior si existe
dataset_path = os.path.join(WORK_DIR, 'Achachairu-1')
if os.path.exists(dataset_path):
    shutil.rmtree(dataset_path)

# Cambiar al directorio de trabajo de Kaggle
os.chdir(WORK_DIR)

rf = Roboflow(api_key="iAo0Rm2Dwt2m0yvaCcBm")
project = rf.workspace("santiagos-workspace-2quku").project("achachairu-bh5vl")
version = project.version(1)
dataset = version.download("coco")

print(f"Dataset en: {dataset.location}")

TRAIN_DIR  = os.path.join(dataset.location, 'train')
VALID_DIR  = os.path.join(dataset.location, 'valid')
TEST_DIR   = os.path.join(dataset.location, 'test')
TRAIN_JSON = os.path.join(TRAIN_DIR, '_annotations.coco.json')
VALID_JSON = os.path.join(VALID_DIR, '_annotations.coco.json')
TEST_JSON  = os.path.join(TEST_DIR,  '_annotations.coco.json')

In [ ]:
# Verificar clases en el dataset
with open(TRAIN_JSON) as f:
    coco = json.load(f)

print("Clases encontradas:")
for cat in coco['categories']:
    print(f"  id={cat['id']}  nombre='{cat['name']}'")
print(f"\nTotal imágenes train: {len(coco['images'])}")
print(f"Total anotaciones train: {len(coco['annotations'])}")

In [ ]:
class COCODataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, json_path, transforms=None):
        self.img_dir = img_dir
        self.transforms = transforms

        with open(json_path) as f:
            coco = json.load(f)

        # Clases válidas — excluir 'fruit'
        clases_validas = {cat['id']: cat['name']
                          for cat in coco['categories']
                          if cat['name'] != 'fruit'}

        # Remap ids: Buena → 1, Mala → 2
        self.id_remap = {}
        for nuevo_id, viejo_id in enumerate(sorted(clases_validas.keys()), start=1):
            self.id_remap[viejo_id] = nuevo_id

        print(f"Remap de clases: { {clases_validas[k]: v for k, v in self.id_remap.items()} }")

        self.imgs    = {img['id']: img for img in coco['images']}
        self.img_ids = list(self.imgs.keys())

        # Agrupar anotaciones — solo clases válidas
        self.anns = {img_id: [] for img_id in self.img_ids}
        for ann in coco['annotations']:
            if ann['category_id'] in clases_validas:
                self.anns[ann['image_id']].append(ann)

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id   = self.img_ids[idx]
        img_info = self.imgs[img_id]

        img_path = os.path.join(self.img_dir, img_info['file_name'])
        image    = Image.open(img_path).convert('RGB')
        image    = TF.to_tensor(image)

        boxes  = []
        labels = []

        for ann in self.anns[img_id]:
            x, y, w, h = ann['bbox']
            xmin, ymin, xmax, ymax = x, y, x + w, y + h

            if xmax <= xmin or ymax <= ymin:
                continue

            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(self.id_remap[ann['category_id']])

        if len(boxes) == 0:
            boxes  = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,),   dtype=torch.int64)
        else:
            boxes  = torch.tensor(boxes,  dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            'boxes':    boxes,
            'labels':   labels,
            'image_id': torch.tensor([img_id])
        }

        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


train_dataset = COCODataset(TRAIN_DIR, TRAIN_JSON)
valid_dataset = COCODataset(VALID_DIR, VALID_JSON)

# Kaggle tiene limitaciones de workers — usar 2 es seguro
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

print(f'Train: {len(train_dataset)} imágenes')
print(f'Valid: {len(valid_dataset)} imágenes')

In [ ]:
# Faster R-CNN con ResNet50 + FPN
model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

model.to(device)

print(f'Device: {device}')
print(f'Modelo en GPU: {next(model.parameters()).is_cuda}')
print(f'Parámetros totales: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=STEP_LR,
    gamma=GAMMA_LR
)

history = {
    'train_loss': [],
    'val_map50':  [],
    'val_map':    [],
    'val_recall': [],
    'val_f1':     []
}

for epoch in range(EPOCHS):

    # --- ENTRENAMIENTO ---
    model.train()
    epoch_loss = 0.0

    for images, targets in train_loader:
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()

    lr_scheduler.step()
    avg_loss = epoch_loss / len(train_loader)
    history['train_loss'].append(avg_loss)

    # --- VALIDACIÓN ---
    model.eval()
    metric = MeanAveragePrecision(iou_thresholds=[0.5], extended_summary=True)

    with torch.no_grad():
        for images, targets in valid_loader:
            images      = [img.to(device) for img in images]
            preds       = model(images)
            preds_cpu   = [{k: v.cpu() for k, v in p.items()} for p in preds]
            targets_cpu = [{k: v.cpu() for k, v in t.items()} for t in targets]
            metric.update(preds_cpu, targets_cpu)

    results = metric.compute()
    map50   = results['map_50'].item()
    map_val = results['map'].item()
    recall  = results['mar_100'].item()
    f1      = (2 * map50 * recall) / (map50 + recall + 1e-7)

    history['val_map50'].append(map50)
    history['val_map'].append(map_val)
    history['val_recall'].append(recall)
    history['val_f1'].append(f1)

    print(f'Epoch [{epoch+1}/{EPOCHS}]  Loss: {avg_loss:.4f}  '
          f'mAP50: {map50:.4f}  mAP: {map_val:.4f}  '
          f'Recall: {recall:.4f}  F1: {f1:.4f}  '
          f'LR: {lr_scheduler.get_last_lr()[0]:.6f}')

    # Guardar checkpoint cada 10 epochs por si Kaggle corta la sesión
    if (epoch + 1) % 10 == 0:
        ckpt_path = os.path.join(WORK_DIR, f'ckpt_epoch{epoch+1}.pth')
        torch.save(model.state_dict(), ckpt_path)
        print(f'  → Checkpoint guardado: {ckpt_path}')

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(epochs_range, history['train_loss'], color='steelblue')
axes[0,0].set_title('Train Loss')
axes[0,0].set_xlabel('Epoch')
axes[0,0].grid(True)

axes[0,1].plot(epochs_range, history['val_map50'], color='green', label='mAP50')
axes[0,1].plot(epochs_range, history['val_map'],   color='olive', label='mAP50-95')
axes[0,1].set_title('mAP')
axes[0,1].set_xlabel('Epoch')
axes[0,1].legend()
axes[0,1].grid(True)

axes[1,0].plot(epochs_range, history['val_recall'], color='orange')
axes[1,0].set_title('Recall')
axes[1,0].set_xlabel('Epoch')
axes[1,0].grid(True)

axes[1,1].plot(epochs_range, history['val_f1'], color='red')
axes[1,1].set_title('F1 Score')
axes[1,1].set_xlabel('Epoch')
axes[1,1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'metricas.png'), dpi=150)
plt.show()

print(f'\nMejor mAP50:  {max(history["val_map50"]):.4f}  (epoch {history["val_map50"].index(max(history["val_map50"]))+1})')
print(f'Mejor mAP:    {max(history["val_map"]):.4f}')
print(f'Mejor Recall: {max(history["val_recall"]):.4f}')
print(f'Mejor F1:     {max(history["val_f1"]):.4f}')

In [ ]:
# Evaluación final en test
test_dataset = COCODataset(TEST_DIR, TEST_JSON)
test_loader  = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

model.eval()
metric_test = MeanAveragePrecision(iou_thresholds=[0.5], extended_summary=True)

with torch.no_grad():
    for images, targets in test_loader:
        images      = [img.to(device) for img in images]
        preds       = model(images)
        preds_cpu   = [{k: v.cpu() for k, v in p.items()} for p in preds]
        targets_cpu = [{k: v.cpu() for k, v in t.items()} for t in targets]
        metric_test.update(preds_cpu, targets_cpu)

r = metric_test.compute()
map50_test  = r['map_50'].item()
map_test    = r['map'].item()
recall_test = r['mar_100'].item()
f1_test     = (2 * map50_test * recall_test) / (map50_test + recall_test + 1e-7)

print('=== Evaluación en TEST ===')
print(f'AP50:    {map50_test:.4f}')
print(f'mAP:     {map_test:.4f}')
print(f'Recall:  {recall_test:.4f}')
print(f'F1:      {f1_test:.4f}')

In [ ]:
# Guardar modelo final en /kaggle/working para descargarlo
model_path = os.path.join(WORK_DIR, 'fasterrcnn_resnet50.pth')
torch.save(model.state_dict(), model_path)
print(f'Modelo guardado: {model_path}')